# ERCC Analysis 

This jupyter notebook runs ERCC analysis for the GeneLab RNASeq pipeline using a combination of python for manipulating and plotting the data and R for DESeq2 analysis.

To run the ERCC Analysis, edit the input variables as specified below and execute each cell in the notebook. Make sure to switch kernels where specified.

<br>

The notebook has 3 sections:
- **Step 1:** Evaluate ERCC Count Data in Python
- **Step 2:** Perform DESeq2 Analysis of ERCC Counts in R
- **Step 3:** Analyze ERCC DESeq2 Results in Python
> Ensure that the correct kernel is chosen at the beginning of each step 

<br>

The first cell in each section contains input variables that need to be updated to match the input data.
- **Step 1:**
    - `accession` - the accession number for the OSDR study. Used to set the output filenames.
    - `isaPath` - the filepath for the ISA.zip file, a compressed ISA directory containing Investigation, Study, and Assay (ISA) metadata files for the respective OSDR dataset. Sample (`s_*.txt`) and RNA-seq assay tables are selected automatically from the archive.
    - `UnnormalizedCountsPath` - the file path for the RSEM Unnormalized Counts file. Created in the "Export DGE Tables" Step of the GeneLab RNAseq Consensus Pipeline. (Default: `RSEM_Unnormalized_Counts{assay_suffix}.csv`)
    - `assay_suffix` - the assay suffix used in the filenames. (Default: `_GLbulkRNAseq`)
    - `GENE_ID_PREFIX` - the common prefix for all Gene IDs associated with the subject organism (default: `EN`, Ensembl)
- **Step 2:**
    - `assay_suffix` - the assay suffix used in the filenames. (Default: `_GLbulkRNAseq`)
- **Step 3:**
    - `accession` - the accession number for the OSDR study. Used to set the output filenames.
    - `assay_suffix` - the assay suffix used in the filenames. (Default: `_GLbulkRNAseq`)

# Step 1: Evaluate ERCC Count Data in Python

> Uses **python** kernel

In [ ]:
import os
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import linregress

pd.set_option('mode.chained_assignment', None)

accession = 'GLDS-NNN'
isaPath = '/path/to/Metadata/OSD-NNN_metadata_OSD-NNN-ISA.zip'
assay_suffix = '_GLbulkRNAseq'
UnnormalizedCountsPath = f'/path/to/RSEM_Counts/RSEM_Unnormalized_Counts{assay_suffix}.csv'
GENE_ID_PREFIX = 'EN'
ercc_url = os.environ.get('ERCC_TABLE', 'https://assets.thermofisher.com/TFS-Assets/LSG/manuals/cms_095046.txt')

zip_file_object = zipfile.ZipFile(isaPath)
list_of_ISA_files = zip_file_object.namelist()

In [ ]:
RNASEQ_MEASUREMENT = "transcription profiling"
RNASEQ_TECHNOLOGY = "RNA Sequencing (RNA-Seq)"

def _is_section_header(line):
    return bool(line) and "\t" not in line and line == line.upper() and any(c.isalpha() for c in line)

def _isa_section(text, header):
    table = {}
    in_section = False
    for raw in text.splitlines():
        line = raw.strip()
        if line == header:
            in_section = True
            continue
        if in_section and (not line or (_is_section_header(line) and line != header)):
            break
        if in_section:
            parts = line.split("\t")
            if parts and parts[0]:
                table[parts[0]] = [v.strip() for v in parts[1:]]
    return table

try:
    i_file = next(m for m in list_of_ISA_files if Path(m).name.lower().startswith("i_") and m.endswith(".txt"))
    assays = _isa_section(zip_file_object.read(i_file).decode("utf-8"), "STUDY ASSAYS")
    assay_name = next(
        fn
        for meas, tech, fn in zip(
            assays["Study Assay Measurement Type"],
            assays["Study Assay Technology Type"],
            assays["Study Assay File Name"],
        )
        if meas.lower() == RNASEQ_MEASUREMENT.lower() and tech.lower() == RNASEQ_TECHNOLOGY.lower()
    )
    sample_file = next(m for m in list_of_ISA_files if Path(m).name.startswith("s_") and m.endswith(".txt"))
    assay_file = next(m for m in list_of_ISA_files if Path(m).name == Path(assay_name).name)
except (StopIteration, KeyError) as e:
    print(f"Metadata selection failed: {e or type(e).__name__}")
    for i, m in enumerate(list_of_ISA_files):
        print(i, m)
    sample_file = list_of_ISA_files[int(input("sample file index: "))]
    assay_file = list_of_ISA_files[int(input("assay file index: "))]
print(sample_file)
print(assay_file)

sample_table = pd.read_csv(zip_file_object.open(sample_file), sep='\t')
assay_table = pd.read_csv(zip_file_object.open(assay_file), sep='\t')

pd.set_option('display.max_columns', None)
print(sample_table.head(n=3))
assay_table.head(n=3)

### Get raw counts table

In [ ]:
raw_counts_table = pd.read_csv(UnnormalizedCountsPath, index_col=0) 
raw_counts_table.index.rename('Gene_ID', inplace=True)
print(raw_counts_table.head(n=3))

raw_counts_transcripts = raw_counts_table[raw_counts_table.index.str.contains(f"^{GENE_ID_PREFIX}")]
assert len(raw_counts_transcripts) != 0, f"Looks like {GENE_ID_PREFIX} matched no genes, probably the wrong prefix"
raw_counts_transcripts = raw_counts_transcripts.sort_values(by=list(raw_counts_transcripts), ascending=False)
from IPython.display import HTML, display
import html
display(HTML(
    f"<details><summary>{len(raw_counts_transcripts)} host genes</summary>"
    f"<pre>{html.escape(raw_counts_transcripts.to_string())}</pre></details>"
))

### Get ERCC counts

In [ ]:
ercc_counts = raw_counts_table[raw_counts_table.index.str.contains('^ERCC-')] 
ercc_counts.reset_index(inplace=True)
ercc_counts = ercc_counts.rename(columns={'Gene_ID':'ERCC ID'})
ercc_counts = ercc_counts.sort_values(by=list(ercc_counts), ascending=False)
print(ercc_counts.head())

### Get files containing ERCC gene concentrations and metadata

In [ ]:
ercc_table = pd.read_csv(ercc_url, sep = '\t')
print(ercc_table.head(n=3))

## Calculate the number of ERCC genes detected in each of the 4 (A, B, C and D) groups for each sample
### Extract ERCC counts


In [ ]:
meltERCC = ercc_counts.melt(id_vars=['ERCC ID'])
meltERCC = meltERCC.rename(columns={'variable':'Sample Name', 'value':'Count'})
print(meltERCC.head(n=3))

### Build Mix dictionary to link sample name to mix added and read depth using the assay table


In [ ]:
mix_dict = assay_table.filter(['Sample Name','Parameter Value[Spike-in Mix Number]', 
                       'Parameter Value[Read Depth]'])
mix_dict = mix_dict.rename(columns={'Parameter Value[Spike-in Mix Number]':'Mix',
                                    'Parameter Value[Read Depth]':
                                    'Total Reads'})
print(mix_dict.head(n=3))

### Make combined ercc counts and assay table


In [ ]:
merged_ercc = meltERCC.merge(mix_dict, on='Sample Name')
print(merged_ercc)


### Read ERCC info including concentrations from merged_ercc table

In [ ]:
groupA = ercc_table.loc[ercc_table['subgroup'] == 'A']['ERCC ID']
groupB = ercc_table.loc[ercc_table['subgroup'] == 'B']['ERCC ID']
groupC = ercc_table.loc[ercc_table['subgroup'] == 'C']['ERCC ID']
groupD = ercc_table.loc[ercc_table['subgroup'] == 'D']['ERCC ID']

### Make a dictionary for ERCC groups

In [ ]:
group_dict = dict(zip(ercc_table['ERCC ID'], ercc_table['subgroup']))

### Calculate ERCC counts per million and log(2) counts per million

In [ ]:
merged_ercc['Count per million'] = merged_ercc['Count'] / (merged_ercc['Total Reads'] / 1000000.0)
merged_ercc['log2 Count per million'] = np.log2(merged_ercc['Count per million']+1)

### Add ERCC group column

In [ ]:
merged_ercc['ERCC group'] = merged_ercc['ERCC ID'].map(group_dict)
merged_ercc = merged_ercc.sort_values(by=['Mix'], ascending=True)
print(merged_ercc)

## Filter and calculate mean counts per million of Mix1 and Mix2 spiked samples in each of the 4 groups
### Filter Mix1 CPM and Mix2 CPM in group A 

In [ ]:
Adf = merged_ercc.loc[merged_ercc['ERCC group'] == 'A']
Amix1df = Adf.loc[Adf['Mix']=='Mix 1']
Amix1df['Mix1 CPM'] = Amix1df[Amix1df['Count per million'] > 0]['Count per million'].dropna()
Amix1df = Amix1df.groupby('ERCC ID')['Mix1 CPM'].agg('mean').rename('Avg Mix1 CPM')
Amix1df = Amix1df.to_frame()
Amix2df = Adf.loc[Adf['Mix']=='Mix 2']
Amix2df['Mix2 CPM'] = Amix2df[Amix2df['Count per million'] > 0]['Count per million'].dropna()
Amix2df = Amix2df.groupby('ERCC ID')['Mix2 CPM'].agg('mean').rename('Avg Mix2 CPM')
Amix2df = Amix2df.to_frame()

adf = Amix1df.merge(Amix2df, on='ERCC ID', suffixes=('', '_2'))
adf = adf.reset_index()
adf['Avg Mix1 CPM/ Avg Mix2 CPM'] = (adf['Avg Mix1 CPM'] / adf['Avg Mix2 CPM'])

### Filter Mix1 CPM and Mix2 CPM in group B

In [ ]:
Bdf = merged_ercc.loc[merged_ercc['ERCC group'] == 'B']
Bmix1df = Bdf.loc[Bdf['Mix']=='Mix 1']
Bmix1df['Mix1 CPM'] = Bmix1df[Bmix1df['Count per million'] > 0]['Count per million'].dropna()
Bmix1df = Bmix1df.groupby('ERCC ID')['Mix1 CPM'].agg('mean').rename('Avg Mix1 CPM')
Bmix1df = Bmix1df.to_frame()
Bmix2df = Bdf.loc[Bdf['Mix']=='Mix 2']
Bmix2df['Mix2 CPM'] = Bmix2df[Bmix2df['Count per million'] > 0]['Count per million'].dropna()
Bmix2df = Bmix2df.groupby('ERCC ID')['Mix2 CPM'].agg('mean').rename('Avg Mix2 CPM')
Bmix2df = Bmix2df.to_frame()

bdf = Bmix1df.merge(Bmix2df, on='ERCC ID')
bdf = bdf.reset_index()
bdf['Avg Mix1 CPM/ Avg Mix2 CPM'] = (bdf['Avg Mix1 CPM'] / bdf['Avg Mix2 CPM'])

### Filter Mix1 CPM and Mix2 CPM in group C

In [ ]:
Cdf = merged_ercc.loc[merged_ercc['ERCC group'] == 'C']
Cmix1df = Cdf.loc[Cdf['Mix']=='Mix 1']
Cmix1df['Mix1 CPM'] = Cmix1df[Cmix1df['Count per million'] > 0]['Count per million'].dropna()
Cmix1df = Cmix1df.groupby('ERCC ID')['Mix1 CPM'].agg('mean').rename('Avg Mix1 CPM')
Cmix1df = Cmix1df.to_frame()
Cmix2df = Cdf.loc[Cdf['Mix']=='Mix 2']
Cmix2df['Mix2 CPM'] = Cmix2df[Cmix2df['Count per million'] > 0]['Count per million'].dropna()
Cmix2df = Cmix2df.groupby('ERCC ID')['Mix2 CPM'].agg('mean').rename('Avg Mix2 CPM')
Cmix2df = Cmix2df.to_frame()

cdf = Cmix1df.merge(Cmix2df, on='ERCC ID')
cdf = cdf.reset_index()
cdf['Avg Mix1 CPM/ Avg Mix2 CPM'] = (cdf['Avg Mix1 CPM'] / cdf['Avg Mix2 CPM'])

### Filter Mix1 CPM and Mix2 CPM in group D

In [ ]:
Ddf = merged_ercc.loc[merged_ercc['ERCC group'] == 'D']
Dmix1df = Ddf.loc[Ddf['Mix']=='Mix 1']
Dmix1df['Mix1 CPM'] = Dmix1df[Dmix1df['Count per million'] > 0]['Count per million'].dropna()
Dmix1df = Dmix1df.groupby('ERCC ID')['Mix1 CPM'].agg('mean').rename('Avg Mix1 CPM')
Dmix1df = Dmix1df.to_frame()
Dmix2df = Ddf.loc[Ddf['Mix']=='Mix 2']
Dmix2df['Mix2 CPM'] = Dmix2df[Dmix2df['Count per million'] > 0]['Count per million'].dropna()
Dmix2df = Dmix2df.groupby('ERCC ID')['Mix2 CPM'].agg('mean').rename('Avg Mix2 CPM')
Dmix2df = Dmix2df.to_frame()

ddf = Dmix1df.merge(Dmix2df, on='ERCC ID')
ddf = ddf.reset_index()
ddf['Avg Mix1 CPM/ Avg Mix2 CPM'] = (ddf['Avg Mix1 CPM'] / ddf['Avg Mix2 CPM'])

## Multi-sample ERCC analyses
### Create box and whisker plots of the log(2) CPM for each ERCC detected in group A in Mix 1 and Mix 2 spiked samples

In [ ]:
a = sns.catplot(x="ERCC ID", y="log2 Count per million", order=groupA[::-1], hue="Mix", data=merged_ercc[merged_ercc['ERCC ID'].isin(groupA)], kind="box", col="ERCC group", height=5, aspect=1, palette=sns.color_palette(['blue', 'orange']), showfliers=False)
a.set_xticklabels(rotation=90)
plt.text(23,2.5,"Mix1/ Mix2 = 4")

### Create bar plot of the average Mix1 CPM / average Mix 2 CPM for group A ERCC genes (for group A we expect Mix 1 CPM / Mix 2 CPM = 4)

In [ ]:
a1 = sns.catplot(x="ERCC ID", y="Avg Mix1 CPM/ Avg Mix2 CPM", order=groupA[::-1], hue="ERCC ID", hue_order=groupA[::-1], 
                 palette="rocket_r", data=adf, kind="bar", height=5, aspect=1, linewidth=0.5)
a1.set_xticklabels(rotation=90)
plt.title("ERCC Group A")
a1.set(ylim=(0, 6))
a1.set_axis_labels("ERCC genes ordered by concentration: low \u2192 high")
print('Number of ERCC detected in group A (out of 23) =', adf['Avg Mix1 CPM/ Avg Mix2 CPM'].count())

### Create box and whisker plots of the log(2) CPM for each ERCC detected in group B in Mix 1 and Mix 2 spiked samples

In [ ]:
b = sns.catplot(x="ERCC ID", y="log2 Count per million", order=groupB[::-1], hue="Mix", data=merged_ercc[merged_ercc['ERCC ID'].isin(groupB)], kind="box", col="ERCC group", height=5, aspect=1, palette=sns.color_palette(['blue', 'orange']), showfliers=False)
b.set_xticklabels(rotation=90)
plt.text(23,2.5,"Mix1/ Mix2 = 1")

### Create bar plot of the average Mix1 CPM / average Mix 2 CPM for group B ERCC genes (for group B we expect Mix 1 CPM / Mix 2 CPM = 1)

In [ ]:
b = sns.catplot(x="ERCC ID", y="Avg Mix1 CPM/ Avg Mix2 CPM", order=groupB[::-1], hue="ERCC ID", hue_order=groupB[::-1], 
                palette="rocket_r", data=bdf, kind="bar", height=5, aspect=1, linewidth=0.5)
b.set_xticklabels(rotation=90)
plt.title("ERCC Group B")
b.set(ylim=(0, 2))
b.set_axis_labels("ERCC genes ordered by concentration: low \u2192 high")
print('Number of ERCC detected in group B (out of 23) =', bdf['Avg Mix1 CPM/ Avg Mix2 CPM'].count())

### Create box and whisker plots of the log(2) CPM for each ERCC detected in group C in Mix 1 and Mix 2 spiked samples

In [ ]:
c = sns.catplot(x="ERCC ID", y="log2 Count per million", order=groupC[::-1], hue="Mix", data=merged_ercc[merged_ercc['ERCC ID'].isin(groupC)], kind="box", col="ERCC group", height=5, aspect=1, palette=sns.color_palette(['blue', 'orange']), showfliers=False)
c.set_xticklabels(rotation=90)
plt.text(23,2.5,"Mix1/ Mix2 = 0.67")

### Create bar plot of the average Mix1 CPM / average Mix 2 CPM for group C ERCC genes (for group C we expect Mix 1 CPM / Mix 2 CPM = 0.67)

In [ ]:
c = sns.catplot(x="ERCC ID", y="Avg Mix1 CPM/ Avg Mix2 CPM", order=groupC[::-1], hue="ERCC ID", hue_order=groupC[::-1], 
                palette="rocket_r", data=cdf, kind="bar", height=5, aspect=1, linewidth=0.5)
c.set_xticklabels(rotation=90)
plt.title("ERCC Group C")
c.set(ylim=(0, 2))
c.set_axis_labels("ERCC genes ordered by concentration: low \u2192 high")
print('Number of ERCC detected in group C (out of 23) =', cdf['Avg Mix1 CPM/ Avg Mix2 CPM'].count())

### Create box and whisker plots of the log(2) CPM for each ERCC detected in group D in Mix 1 and Mix 2 spiked samples

In [ ]:
d = sns.catplot(x="ERCC ID", y="log2 Count per million", order=groupD[::-1], hue="Mix", data=merged_ercc[merged_ercc['ERCC ID'].isin(groupD)], col="ERCC group", kind="box", height=5, aspect=1, palette=sns.color_palette(['blue', 'orange']), showfliers=False)
d.set_xticklabels(rotation=90)
plt.text(23,2.5,"Mix1/ Mix2 = 0.5")

### Create bar plot of the average Mix1 CPM / average Mix 2 CPM for group D ERCC genes (for group D we expect Mix 1 CPM / Mix 2 CPM = 0.5)

In [ ]:
d = sns.catplot(x="ERCC ID", y="Avg Mix1 CPM/ Avg Mix2 CPM", order=groupD[::-1], hue="ERCC ID", hue_order=groupD[::-1],
                palette="rocket_r", data=ddf, kind="bar", height=5, aspect=1, linewidth=0.5)
d.set_xticklabels(rotation=90)
plt.title("ERCC Group D")
d.set(ylim=(0, 1))
d.set_axis_labels("ERCC genes ordered by concentration: low \u2192 high")
print('Number of ERCC detected in group D (out of 23) =', ddf['Avg Mix1 CPM/ Avg Mix2 CPM'].count())

## Individual sample ERCC analyses

Calculate and plot ERCC metrics from individual samples, including limit of detection, dynamic range, and R (Pearson r) of counts vs. concentration.

In [ ]:
#Calculate and plot ERCC metrics from individual samples, including limit of detection, dynamic range, and R (Pearson r) of counts vs. concentration.
print(ercc_table.head(n=3))

# Make a dictionary for ERCC concentrations for each mix

mix1_conc_dict = dict(zip(ercc_table['ERCC ID'], ercc_table['concentration in Mix 1 (attomoles/ul)']))
mix2_conc_dict = dict(zip(ercc_table['ERCC ID'], ercc_table['concentration in Mix 2 (attomoles/ul)']))

# Check assay_table header to identify the 'Sample Name' column and the column title indicating the 'Spike-in Mix Number' if it's indicated in the metadata.

pd.set_option('display.max_columns', None)
print(assay_table.head(n=3))

# Get samples that use mix 1 and mix 2

mix1_samples = assay_table[assay_table['Parameter Value[Spike-in Mix Number]'] == 'Mix 1']['Sample Name']
mix2_samples = assay_table[assay_table['Parameter Value[Spike-in Mix Number]'] == 'Mix 2']['Sample Name']

# Get ERCC counts for all samples

ercc_counts = raw_counts_table[raw_counts_table.index.str.contains('^ERCC-')] 
ercc_counts = ercc_counts.sort_values(by=list(ercc_counts), ascending=False)
print(ercc_counts.head())

# Get ERCC counts for Mix 1 spiked samples

ercc_counts_mix_1 = ercc_counts[mix1_samples]
ercc_counts_mix_1['ERCC conc (attomoles/ul)'] = ercc_counts_mix_1.index.map(mix1_conc_dict)
print(ercc_counts_mix_1.head(n=3))

# Get ERCC counts for Mix 2 spiked samples

ercc_counts_mix_2 = ercc_counts[mix2_samples]
ercc_counts_mix_2['ERCC conc (attomoles/ul)'] = ercc_counts_mix_2.index.map(mix2_conc_dict)
print(ercc_counts_mix_2.head(n=3))


# Create a scatter plot of log(2) ERCC counts versus log(2) ERCC concentration for each sample

columns_mix_1 = ercc_counts_mix_1.columns.drop(['ERCC conc (attomoles/ul)'])
columns_mix_2 = ercc_counts_mix_2.columns.drop(['ERCC conc (attomoles/ul)'])
all_columns = columns_mix_1.to_list() + columns_mix_2.to_list()
total_columns = len(columns_mix_1) + len(columns_mix_2) 
side_size = np.int32(np.ceil(np.sqrt(total_columns)))# calculate grid side size. take sqrt of total plots and round up.
fig, axs = plt.subplots(side_size, side_size, figsize=(22,26), sharex='all', sharey='all'); #change figsize x,y labels if needed.
fig.tight_layout(pad=1, w_pad=2.5, h_pad=3.5)

# Iterate over subplot positions, if subplot does not have data, hide the subplot with ax.set_visible(False)

for i in range(side_size):
    for j in range(side_size):
        ax = axs[i, j]
        index = i * side_size + j

        if index < total_columns:
            if index < len(columns_mix_1):
                ax.scatter(x=np.log2(ercc_counts_mix_1['ERCC conc (attomoles/ul)']), y=np.log2(ercc_counts_mix_1[all_columns[index]]+1), s=7)
                ax.set_title(all_columns[index][-45:], fontsize=9)
            else:
                ax.scatter(x=np.log2(ercc_counts_mix_2['ERCC conc (attomoles/ul)']), y=np.log2(ercc_counts_mix_2[all_columns[index]]+1), s=7)
                ax.set_title(all_columns[index][-45:], fontsize=9)

            ax.set_xlabel('log2 ERCC conc (attomoles/ ul)', fontsize=9)
            ax.set_ylabel('log2(counts+1)', fontsize=9)
            ax.tick_params(direction='in', axis='both', labelsize=9, labelleft=True, labelbottom=True)
        else:
            ax.set_visible(False)  # Hide the subplot if it's not needed

plt.show()

## Calculate and plot linear regression of log(2) ERCC counts versus log(2) ERCC concentration for each sample

In [ ]:
# Filter counts > 0

nonzero_counts_list_1 = []
for i in range(0, len(ercc_counts_mix_1.columns)-1):
  counts = ercc_counts_mix_1[columns_mix_1[i]]
  counts.index.rename('Gene_ID', inplace=True)
  countsdf = pd.DataFrame(counts)
  nonzero_counts = countsdf[ercc_counts_mix_1[columns_mix_1[i]] > 0.0]
  nonzero_counts['Conc'] = nonzero_counts.index.map(mix1_conc_dict)
  nonzero_counts.columns = ['Counts','Conc']
  nonzero_counts_sorted = nonzero_counts.sort_values('Conc')
  nonzero_counts_list_1.append(nonzero_counts_sorted)

nonzero_counts_list_2 = []
for i in range(0, len(ercc_counts_mix_2.columns)-1):
  counts = ercc_counts_mix_2[columns_mix_2[i]]
  counts.index.rename('Gene_ID', inplace=True)
  countsdf = pd.DataFrame(counts)
  nonzero_counts = countsdf[ercc_counts_mix_2[columns_mix_2[i]] > 0.0]
  nonzero_counts['Conc'] = nonzero_counts.index.map(mix2_conc_dict)
  nonzero_counts.columns = ['Counts','Conc']
  nonzero_counts_sorted = nonzero_counts.sort_values('Conc')
  nonzero_counts_list_2.append(nonzero_counts_sorted)


# Plot each sample using linear regression of scatter plot with x = log2 Conc and y = log2 Counts.
# Return min, max, R (Pearson r) and dynamic range (max / min) values.

samples = []
mins = []
maxs = []
dyranges = []
rs = []

fig, axs = plt.subplots(side_size, side_size, figsize=(22,26), sharex='all', sharey='all');
fig.tight_layout(pad=1, w_pad=2.5, h_pad=3.5)

counter = 0
list2counter = 0
for i in range(side_size):
    for j in range(side_size):
        ax = axs[i, j]
        index = i * side_size + j

        if index < len(columns_mix_1):
            nonzero_counts = nonzero_counts_list_1[index]
            xvalues = nonzero_counts['Conc']
            yvalues = nonzero_counts['Counts']

            if len(xvalues) > 0:
                sns.regplot(x=np.log2(xvalues), y=np.log2(yvalues), ax=ax)
                ax.set_title(all_columns[index][-47:], fontsize=9)
                ax.set_xlabel('log2 Conc (attomoles/ul)', fontsize=9)
                ax.set_ylabel('log2 Counts', fontsize=9)
                ax.tick_params(direction='in', axis='both', labelsize=9, labelleft=True, labelbottom=True)
                samples.append(all_columns[index])

                min_val = xvalues.min()
                max_val = xvalues.max()
                dynamic_range = max_val / min_val
                r_value = linregress(np.log2(xvalues), np.log2(yvalues)).rvalue

                ax.text(0.02, 0.98, f'Min:{min_val:.1f}', verticalalignment='top', horizontalalignment='left', transform=ax.transAxes, color='black', fontsize=10)
                ax.text(0.02, 0.88, f'Max:{max_val:.1f}', verticalalignment='top', horizontalalignment='left', transform=ax.transAxes, color='black', fontsize=10)
                ax.text(0.02, 0.78, f'Dyn:{dynamic_range:.1f}', verticalalignment='top', horizontalalignment='left', transform=ax.transAxes, color='black', fontsize=10)
                ax.text(0.02, 0.68, f'R:{r_value:.2f}', verticalalignment='top', horizontalalignment='left', transform=ax.transAxes, color='black', fontsize=10)

                mins.append(min_val)
                maxs.append(max_val)
                dyranges.append(dynamic_range)
                rs.append(r_value)
            else:
                ax.set_visible(False)

        elif index < total_columns:
            nonzero_counts = nonzero_counts_list_2[list2counter]
            xvalues = nonzero_counts['Conc']
            yvalues = nonzero_counts['Counts']

            if len(xvalues) > 0:
                sns.regplot(x=np.log2(xvalues), y=np.log2(yvalues), ax=ax)
                ax.set_title(all_columns[index][-47:], fontsize=9)
                ax.set_xlabel('log2 Conc (attomoles/ul)', fontsize=9)
                ax.set_ylabel('log2 Counts', fontsize=9)
                ax.tick_params(direction='in', axis='both', labelsize=9, labelleft=True, labelbottom=True)
                samples.append(all_columns[index])

                min_val = xvalues.min()
                max_val = xvalues.max()
                dynamic_range = max_val / min_val
                r_value = linregress(np.log2(xvalues), np.log2(yvalues)).rvalue

                ax.text(0.02, 0.98, f'Min:{min_val:.1f}', verticalalignment='top', horizontalalignment='left', transform=ax.transAxes, color='black', fontsize=10)
                ax.text(0.02, 0.88, f'Max:{max_val:.1f}', verticalalignment='top', horizontalalignment='left', transform=ax.transAxes, color='black', fontsize=10)
                ax.text(0.02, 0.78, f'Dyn:{dynamic_range:.1f}', verticalalignment='top', horizontalalignment='left', transform=ax.transAxes, color='black', fontsize=10)
                ax.text(0.02, 0.68, f'R:{r_value:.2f}', verticalalignment='top', horizontalalignment='left', transform=ax.transAxes, color='black', fontsize=10)

                mins.append(min_val)
                maxs.append(max_val)
                dyranges.append(dynamic_range)
                rs.append(r_value)
            else:
                ax.set_visible(False)

            list2counter += 1
        else:
            ax.set_visible(False)

plt.show()


In [ ]:
# Create directory for saved files

os.makedirs(name="ERCC_analysis", exist_ok=True)

In [ ]:
# Print tables containing the dynamic range and R (Pearson r) values for each sample.

stats = pd.DataFrame(list(zip(samples, mins, maxs, dyranges, rs)))
stats.columns = ['Samples', 'Min', 'Max', 'Dynamic range', 'R']
stats.to_csv(f'ERCC_analysis/ERCC_stats_{accession}{assay_suffix}.csv', index = False)
stats.filter(items = ['Samples', 'Dynamic range']).to_csv(f'ERCC_analysis/ERCC_dynrange_{accession}_mqc{assay_suffix}.csv', index = False)
stats.filter(items = ['Samples', 'R']).to_csv(f'ERCC_analysis/ERCC_rsq_{accession}_mqc{assay_suffix}.csv', index = False)

## Generate data and metadata files needed for ERCC DESeq2 analysis

In [ ]:
# ERCC Mix 1 and Mix 2 are distributed so that half the samples receive Mix 1 spike-in and half receive Mix 2 spike-in. Transcripts in Mix 1 and Mix 2 are present at a known ratio, so we can determine how well these patterns are revealed in the dataset.

# Get sample table

combined = sample_table.merge(assay_table, on='Sample Name')
combined = combined.set_index(combined['Sample Name'])

# Create metadata table containing samples and their respective ERCC spike-in Mix number
# Sometimes Number in [Spike-in Mix Number] is spelled 'number' and this could cause error in mismatch search 

ERCCmetadata = combined[['Parameter Value[Spike-in Mix Number]']]
ERCCmetadata.index = ERCCmetadata.index.str.replace('-','_')
ERCCmetadata.columns = ['Mix']
print(ERCCmetadata)

# Export ERCC sample metadata

ERCCmetadata.to_csv(f'ERCC_analysis/ERCCmetadata{assay_suffix}.csv') 

# Export ERCC count data

ercc_counts.columns = ercc_counts.columns.str.replace('-','_')
ERCCcounts = ercc_counts.loc[:,ERCCmetadata.index]
ERCCcounts.head()

ERCCcounts.to_csv(f'ERCC_analysis/ERCCcounts{assay_suffix}.csv') 

# Step 2: Perform DESeq2 Analysis of ERCC Counts in R

> Uses **R** kernel

## Load DESeq2


In [ ]:
# Update assay_suffix variable, if not default
assay_suffix <- "_GLbulkRNAseq"


In [ ]:
if (!requireNamespace("DESeq2", quietly = TRUE)) {
    if (!requireNamespace("BiocManager", quietly = TRUE)) {
        install.packages("BiocManager")
    }
    BiocManager::install("DESeq2")
}
library("DESeq2")

## Import and format ERCC count data and metadata

In [ ]:
cts <- as.matrix(read.csv(paste0('ERCC_analysis/ERCCcounts',assay_suffix,'.csv'),sep=",",row.names="Gene_ID")) #INPUT
coldata <- read.csv(paste0('ERCC_analysis/ERCCmetadata',assay_suffix,'.csv'), row.names=1) #INPUT

coldata$Mix <- factor(coldata$Mix)
all(rownames(coldata) == colnames(cts))

## Make DESeqDataSet object

In [ ]:
dds <- DESeqDataSetFromMatrix(countData = round(cts),
                              colData = coldata,
                              design = ~ Mix)
dds

## Filter out ERCC genes with counts of less than 10 in all samples

In [ ]:
keepGenes <- rowSums(counts(dds)) > 10
dds <- dds[keepGenes,]

dds

## Run DESeq2 analysis and calculate results

In [ ]:
## Try first to use the default type="median", but if there is an error (usually due to zeros in genes), use type="poscounts"
## From DESeq2 manual: "The "poscounts" estimator deals with a gene with some zeros, by calculating a modified geometric mean by taking the n-th root of the product of the non-zero counts."
dds <- tryCatch(
      expr = { estimateSizeFactors(dds) },
      error = function(e) { estimateSizeFactors(dds, type="poscounts")}
)

dds <- DESeq(dds)
res <- results(dds, contrast=c("Mix","Mix 1","Mix 2")) # remove space before mix number if needed
res

## Export DESeq2 results table and normalized ERCC counts table

In [ ]:
write.csv(res, paste0('ERCC_analysis/ERCC_DESeq2',assay_suffix,'.csv')) #OUTPUT
normcounts = counts(dds, normalized=TRUE)
write.csv(normcounts, paste0('ERCC_analysis/ERCC_normcounts',assay_suffix,'.csv')) #OUTPUT

# Step 3:  Analyze ERCC DESeq2 Results in Python

> Uses **python** kernel

## Import python packages


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

accession = 'GLDS-NNN'
assay_suffix = '_GLbulkRNAseq'
ercc_url = os.environ.get('ERCC_TABLE', 'https://assets.thermofisher.com/TFS-Assets/LSG/manuals/cms_095046.txt')

## Import ERCC DESeq2 results

In [ ]:
deseq2out = pd.read_csv(f'ERCC_analysis/ERCC_DESeq2{assay_suffix}.csv', index_col=0) # INPUT
#deseq2out.index = deseq2out.index.str.replace('_','-')
deseq2out.rename(columns ={'baseMean' : 'meanNormCounts'}, inplace = True)
print(deseq2out.head())

## Get files containing ERCC gene concentrations and metadata

In [ ]:
ercc_table = pd.read_csv(ercc_url, sep='\t', index_col='ERCC ID')
print(ercc_table.head(n=3))

## Combine ERCC DESeq2 results and ercc_table

In [ ]:
combined = deseq2out.merge(ercc_table, left_index=True, right_index=True)
print(combined.head())

## Filter p-value and adj. p-value cutoff at 10^-3

In [ ]:
combined['cleaned_padj'] = combined['padj']
combined.loc[(combined.cleaned_padj < 0.001),'cleaned_padj']=0.001

combined['cleaned_pvalue'] = combined['pvalue']
combined.loc[(combined.cleaned_pvalue < 0.001),'cleaned_pvalue']=0.001

print(combined.head())

## Export the filtered combined ERCC DESeq2 results and ercc_table


In [ ]:
combined.filter(items = ['ERCC ID', 'meanNormCounts', 'cleaned_pvalue','cleaned_padj']).to_csv(f'ERCC_analysis/ERCC_lodr_{accession}_mqc{assay_suffix}.csv') 

## Plot p-value vs. mean normalized ERCC counts


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sns.scatterplot(data=combined, x="meanNormCounts", y="cleaned_pvalue",
            hue="expected fold-change ratio",
                palette=['red','green','black','blue'], ax=ax)

sns.lineplot(data=combined, x="meanNormCounts", y="cleaned_pvalue",
            hue="expected fold-change ratio",
                palette=['red','green','black','blue'], ax=ax)

#g.set_xscale("log", base=2)
ax.set_xscale("linear");
ax.set_yscale("log");


## Plot adj. p-value vs. mean normalized ERCC counts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sns.scatterplot(data=combined, x="meanNormCounts", y="cleaned_padj",
            hue="expected fold-change ratio",
                palette=['red','green','black','blue'], ax=ax)

sns.lineplot(data=combined, x="meanNormCounts", y="cleaned_padj",
            hue="expected fold-change ratio",
                palette=['red','green','black','blue'], ax=ax)

#g.set_xscale("log", base=2)
ax.set_xscale("linear");
ax.set_yscale("log");